# Mechanizm pobierania danych z GUS (API DBW)

https://api.stat.gov.pl/Home/DBWApi

https://api-dbw.stat.gov.pl/apidocs/index.html

Pomyślane jako narzędzie do pobrania danych o inflacji (CPI Usługi lekarskie), ale w Main można przejrzeć dane i ew. zmodyfikować filtrowania po nazwie.

## Importy

In [ ]:
from requests.exceptions import HTTPError, Timeout, ConnectionError
import pandas as pd
from datetime import datetime, timedelta
from pathlib import Path
import logging
import os
import time

from dotenv import load_dotenv
from functools import lru_cache
import requests

load_dotenv()

pd.set_option("display.max_columns", None)

In [ ]:
LOG_PATH = Path("data/gus_fetch.log")
LOG_PATH.parent.mkdir(exist_ok=True)

logger = logging.getLogger("gus")
logger.setLevel(logging.INFO)
logger.propagate = False

if logger.handlers:
    logger.handlers.clear()

logger.addHandler(logging.FileHandler(LOG_PATH, encoding="utf-8"))
logger.handlers[0].setFormatter(
    logging.Formatter("%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S")
)

print(f"Log: {LOG_PATH.resolve()}", flush=True)

## Funkcje pomocnicze

In [58]:
def _get_session() -> requests.Session:
    api_key = os.environ["GUS_API_KEY"]
    session = requests.Session()
    session.headers.update({
        "accept": "application/json",
        "X-ClientId": api_key,
    })
    # Brak Retry adaptera — całą logikę ponawiania obsługuje _get()

    def _log_response(response, *args, **kwargs):
        elapsed = response.elapsed.total_seconds()
        endpoint = response.url.split("/api/")[-1].split("?")[0]
        logger.info(f"{endpoint} → {response.status_code} ({elapsed:.2f}s)")

    session.hooks["response"].append(_log_response)
    return session

In [ ]:
QUOTA_WAIT_SECONDS = 15 * 60 + 10
SERVER_ERROR_WAIT_SECONDS = 30

def _wait(reason: str, seconds: int) -> None:
    resume_at = datetime.now() + timedelta(seconds=seconds)
    msg = f"{reason} — czekam {seconds}s, wznawianie o {resume_at:%H:%M:%S}"
    print(msg, flush=True)
    logger.warning(msg)
    for _ in range(seconds):
        time.sleep(1)

In [59]:
class RateLimiter:
    def __init__(self, per_second: int = 10, per_15min: int = 500):
        self._per_second = per_second
        self._per_15min = per_15min
        self._timestamps: list[float] = []

    def _cleanup(self, now: float) -> None:
        self._timestamps = [t for t in self._timestamps if now - t < 900]

    def wait_if_needed(self) -> None:
        now = time.monotonic()
        self._cleanup(now)

        # Limit 500 req / 15 min (z marginesem 5)
        if len(self._timestamps) >= self._per_15min - 5:
            sleep_for = int(900 - (now - self._timestamps[0])) + 1
            _wait("Zbliżam się do limitu 500/15min", sleep_for)
            now = time.monotonic()

        # Limit 10 req / 1 s
        recent_1s = [t for t in self._timestamps if now - t < 1.0]
        if len(recent_1s) >= self._per_second:
            sleep_for = 1.0 - (now - recent_1s[0])
            if sleep_for > 0:
                time.sleep(sleep_for)

        self._timestamps.append(time.monotonic())

    def seconds_until_retry(self) -> int:
        """Zwraca czas oczekiwania po otrzymaniu 429."""
        now = time.monotonic()
        self._cleanup(now)
        recent_1s = [t for t in self._timestamps if now - t < 1.0]
        if len(recent_1s) >= self._per_second:
            return 2  # limit per-second — wystarczy chwila
        return QUOTA_WAIT_SECONDS  # limit 15-min — czekamy długo

In [ ]:
def _get(url: str, params: dict, timeout: tuple = (5, 60)) -> requests.Response:
    """Wrapper na SESSION.get() z obsługą rate limitingu i timeoutów."""
    while True:
        RATE_LIMITER.wait_if_needed()
        try:
            response = SESSION.get(url, params=params, timeout=timeout)
        except (Timeout, ConnectionError) as e:
            _wait(f"Timeout/ConnectionError ({e.__class__.__name__})", QUOTA_WAIT_SECONDS)
            continue

        if response.status_code == 429 or "quota exceeded" in response.text.lower():
            wait_seconds = RATE_LIMITER.seconds_until_retry()
            _wait(f"Rate limit 429 (czekam {wait_seconds}s)", wait_seconds)
            continue

        if response.status_code in (500, 503):
            _wait(f"Błąd serwera ({response.status_code})", SERVER_ERROR_WAIT_SECONDS)
            continue

        return response

### _get_periods

In [61]:
@lru_cache(maxsize=1)
def _get_periods() -> dict:
    """Pobiera słownik okresów statystycznych z API GUS DBW."""
    url = "https://api-dbw.stat.gov.pl/api/dictionaries/periods-dictionary"
    params = {"page-size": 5000, "page": 0, "lang": "pl"}

    first = _get(url, params, timeout=(5, 60))
    first.raise_for_status()
    first_json = first.json()

    pages = first_json["page-count"]
    data = first_json["data"]

    for page in range(1, pages + 1):
        params["page"] = page
        response = _get(url, params, timeout=(5, 60))
        response.raise_for_status()
        data.extend(response.json()["data"])

    return {d["id-okres"]: d["opis"] for d in data}

### get_variables_periods_sections

In [62]:
def get_variables_periods_sections() -> pd.DataFrame:
    """Pobiera pełny zakres zmiennych, przekrojów oraz okresów z API GUS DBW."""
    url = "https://api-dbw.stat.gov.pl/api/variable/variable-section-periods"
    params = {"ile-na-stronie": 5000, "numer-strony": 0, "lang": "pl"}

    first = _get(url, params, timeout=(5, 60))
    first.raise_for_status()
    first_json = first.json()

    pages = first_json["page-count"]
    data = first_json["data"]

    for page in range(1, pages + 1):
        params["numer-strony"] = page
        response = _get(url, params, timeout=(5, 60))
        response.raise_for_status()
        data.extend(response.json()["data"])
        logger.info(f"variable-section-periods: strona {page}/{pages}")

    lookup = _get_periods()
    df = pd.DataFrame(data)
    df["opis-okres"] = df["id-okres"].map(lookup)

    return df

### _get_positions_lookup

In [63]:
@lru_cache(maxsize=128)
def _get_positions_lookup(section_id: int) -> dict:
    """Pobiera słownik pozycji statystycznych dla danego przekroju z API GUS DBW."""
    url = "https://api-dbw.stat.gov.pl/api/variable/variable-section-position"
    params = {"id-przekroj": section_id, "lang": "pl"}

    response = _get(url, params, timeout=(5, 60))
    response.raise_for_status()

    return {d["id-pozycja"]: d["nazwa-pozycja"] for d in response.json()}

### _get_ways_of_presentation

In [64]:
@lru_cache(maxsize=1)
def _get_ways_of_presentation() -> dict:
    """Pobiera słownik sposobów prezentacji z API GUS DBW."""
    url = "https://api-dbw.stat.gov.pl/api/dictionaries/way-of-presentation"
    params = {"page": 0, "page-size": 5000, "lang": "pl"}

    first = _get(url, params, timeout=(5, 60))
    first.raise_for_status()
    first_json = first.json()

    pages = first_json["page-count"]
    data = first_json["data"]

    for page in range(1, pages + 1):
        params["page"] = page
        response = _get(url, params, timeout=(5, 60))
        response.raise_for_status()
        data.extend(response.json()["data"])

    return {d["id-sposob-prezentacji-miara"]: d["nazwa"] for d in data}

### get_positions 

In [65]:
def get_positions(section_ids: list[int]) -> pd.DataFrame:
    """
    Pobiera pozycje statystyczne wraz z informacją o wymiarach
    dla wskazanych przekrojów z API GUS DBW.

    Parameters
    ----------
    section_ids : list[int]
        Identyfikatory przekrojów, dla których mają zostać pobrane pozycje.
    """
    url = "https://api-dbw.stat.gov.pl/api/variable/variable-section-position"

    dfs = []
    for section_id in section_ids:
        params = {"id-przekroj": section_id, "lang": "pl"}
        response = _get(url, params, timeout=(5, 60))
        response.raise_for_status()
        dfs.append(pd.DataFrame(response.json()))

    return pd.concat(dfs, ignore_index=True)

### get_years_to_fetch

In [ ]:
def get_years_to_fetch(parquet_path: str, all_years: list[int]) -> list[int]:
    """Zwraca lata których jeszcze nie ma w parquet."""
    if not Path(parquet_path).exists():
        return all_years

    df_existing = pd.read_parquet(parquet_path, columns=["id-rok"])
    fetched_years = set(df_existing["id-rok"].unique())
    last_fetched = max(fetched_years) if fetched_years else None

    # Zawsze ponownie pobieraj ostatni rok — GUS może korygować dane wstecznie
    return [y for y in all_years if y not in fetched_years or y == last_fetched]

### get_data

In [ ]:
def get_data(
    variable_id: int,
    section_periods: dict[int, list[int]],
    year_ids: list[int],
    section_names: dict,
) -> pd.DataFrame:
    url = "https://api-dbw.stat.gov.pl/api/variable/variable-data-section"
    results = []

    for section_id, period_ids in section_periods.items():
        for year_id in year_ids:
            logger.info(f"Pobieranie: section={section_id}, rok={year_id}, periods={len(period_ids)}")
            for period_id in period_ids:
                params = {
                    "id-zmienna": variable_id,
                    "id-przekroj": section_id,
                    "id-rok": year_id,
                    "id-okres": period_id,
                    "ile-na-stronie": 5000,
                    "numer-strony": 0,
                    "lang": "pl",
                }

                try:
                    first = _get(url, params, timeout=(5, 30))
                    first.raise_for_status()
                except HTTPError as e:
                    if e.response.status_code == 404:
                        continue
                    raise

                first_json = first.json()
                pages = first_json["page-count"]
                data = first_json["data"]

                for page in range(1, pages + 1):
                    params["numer-strony"] = page
                    response = _get(url, params, timeout=(5, 30))
                    response.raise_for_status()
                    data.extend(response.json()["data"])

                section_df = pd.DataFrame(data)
                section_df["id-rok"] = year_id
                results.append(section_df)

    if not results:
        return pd.DataFrame()

    df = pd.concat(results, ignore_index=True)
    logger.info("All done")

    df["opis-okres"] = df["id-okres"].map(_get_periods())
    df["sposob-prezentacji"] = df["id-sposob-prezentacji-miara"].map(_get_ways_of_presentation())
    df["opis-pozycja-2"] = df.apply(
        lambda row: _get_positions_lookup(row["id-przekroj"]).get(row["id-pozycja-2"]), axis=1
    )
    df["opis-pozycja-3"] = df.apply(
        lambda row: _get_positions_lookup(row["id-przekroj"]).get(row["id-pozycja-3"]), axis=1
    )
    df["nazwa-przekroj"] = df["id-przekroj"].map(section_names)

    return df[["nazwa-przekroj", "opis-pozycja-3", "opis-pozycja-2", "opis-okres", "sposob-prezentacji", "id-rok", "wartosc"]]

## Main

### Start sesji

In [68]:
SESSION = _get_session()
RATE_LIMITER = RateLimiter()

### Wybór przekroju i zmiennej

In [69]:
%%time
df = get_variables_periods_sections()

CPU times: total: 281 ms
Wall time: 11.5 s


In [70]:
variable_filter = "Wskaźniki cen towarów i usług konsumpcyjnych"
variable_id = df[df["nazwa-zmienna"]==variable_filter]["id-zmienna"].unique()[0]

In [71]:
section_ids = list(df[df["id-zmienna"]==variable_id]["id-przekroj"].unique())
section_names = dict(zip(
    df[df["id-zmienna"]==variable_id]["id-przekroj"],
    df[df["id-zmienna"]==variable_id]["nazwa-przekroj"]
))

### Wybór okresu dla przekroju i zmiennej

In [72]:
section_periods = (
    df[df["id-zmienna"] == variable_id]
    .groupby("id-przekroj")["id-okres"]
    .apply(list)
    .to_dict()
)

### Wybór pozycji

### Pobierz wskaźniki

In [ ]:
years = list(range(2015, 2026))
years_to_fetch = get_years_to_fetch("data/gus_data.parquet", years)
print(f"Lata do pobrania: {years_to_fetch}")

In [76]:
variable_id = int(variable_id)
section_ids = [int(x) for x in section_ids]
section_periods = {int(k): [int(v) for v in vs] for k, vs in section_periods.items()}
position_ids = [int(x) for x in position_ids]

In [ ]:
df

In [ ]:
df = get_data(
    variable_id=variable_id,
    section_periods=section_periods,
    year_ids=years_to_fetch,
    section_names=section_names,
)

In [ ]:
KEY_COLUMNS = ["opis-pozycja-3", "opis-pozycja-2", "opis-okres", "sposob-prezentacji", "nazwa-przekroj", "id-rok"]
PARQUET_PATH = "data/gus_data.parquet"

if Path(PARQUET_PATH).exists():
    df_existing = pd.read_parquet(PARQUET_PATH)
    df_combined = pd.concat([df_existing, df], ignore_index=True)
    df_combined = df_combined.drop_duplicates(subset=KEY_COLUMNS, keep="last")
else:
    df_combined = df

df_combined.to_parquet(PARQUET_PATH, index=False)
print(f"Zapisano {len(df_combined)} wierszy, {__import__('os').path.getsize(PARQUET_PATH) / 1024**2:.1f} MB")